# PathoVision - tables and results bundle (CPU only)

Builds the tables of stage 23 and `results_bundle.zip` from whatever is already saved in
`pathovision_reanalysis/` on Google Drive. No GPU and no Kaggle token are needed.

**How to use:** *Runtime > Change runtime type > CPU*, then *Runtime > Run all* (about 5-10 min).
The bundle is saved to Drive and downloaded automatically.

In [ ]:
#@title Tables and bundle (CPU)
import os, subprocess, pathlib, shutil
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/pathovision_reanalysis'
REPO = '/content/pathovision_model'
os.environ['REANALYSIS_ROOT'] = ROOT
os.environ['REANALYSIS_DATA_ROOT'] = '/content/pv_data'
os.environ['PYTHONPATH'] = REPO

def run(cmd, cwd=None):
    print(f'\n$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode

if not os.path.exists(REPO):
    run(f'git clone --depth 1 https://github.com/alexroehrs/pathovision_model.git {REPO}')
else:
    run('git pull --ff-only', cwd=REPO)
os.chdir(REPO)
run('pip install -q statsmodels 2>&1 | tail -n 1')
rc = run('python reanalysis/s23_integrity_tables.py')
print('stage 23:', 'ok' if rc == 0 else f'FAILED (exit {rc})')

bundle = pathlib.Path(ROOT) / 'results_bundle'
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(pathlib.Path(ROOT) / 'results', bundle / 'results')
for p in pathlib.Path(ROOT, 'runs').rglob('*'):
    if p.is_file() and (p.suffix in ('.csv', '.json', '.yaml') or p.name.startswith('emb_')):
        dst = bundle / 'runs' / p.relative_to(pathlib.Path(ROOT, 'runs')); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dst)
log = pathlib.Path(ROOT) / 'run_all.log'
if log.exists():
    with open(log, errors='replace') as f, open(bundle / 'run_all_tail.log', 'w') as g:
        g.writelines(f.readlines()[-3000:])
zip_path = shutil.make_archive(str(pathlib.Path(ROOT) / 'results_bundle'), 'zip', bundle)
print('bundle written to Drive:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
done = sorted(str(p.parent.relative_to(pathlib.Path(ROOT, 'runs'))) for p in pathlib.Path(ROOT, 'runs').rglob('done.json'))
print(len(done), 'completed runs')
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('automatic download not possible (', e, '); download it from Google Drive > pathovision_reanalysis')